# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tenuka-R/FlyRank-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one content page's performance on a given day. Grain is client_hash_id, content_hash_id, report_date
Tables: fact_content_daily_performance, monthly_partitioned.
Time window: iterating on month 2026-03.
Verified below: dates span 2026-03-01 to 2026-03-31 with almost 10 million rows.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

fact = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]}
)

print(fact.shape)
print(fact["report_date"].min(), "to", fact["report_date"].max())

(9841378, 31)
2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_avg_position, sessions_organic, word_count, content_type.
Label: gsc_clicks
Context: client_hash_id, content_hash_id, report_date, gsc_data_available.
Excluded:
fact_content_query_90d - different grain
ga4_data_available - confusing and ambigious null values
ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other - I could use these but to keep feature count at 5 I will not


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ga4_data_available dtype:", df["ga4_data_available"].dtype)
print(df["ga4_data_available"].value_counts(dropna=False))
print()

ai_cols = ["ai_chatgpt","ai_perplexity","ai_gemini","ai_copilot","ai_claude","ai_meta","ai_other"]
print(df[ai_cols].isna().mean())

ga4_data_available dtype: object
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64

ai_chatgpt       0.30674
ai_perplexity    0.30674
ai_gemini        0.30674
ai_copilot       0.30674
ai_claude        0.30674
ai_meta          0.30674
ai_other         0.30674
dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain - grouped by client_hash_id, content_hash_id, report_date. No duplicate keys so one row means one content item day.
Total records: 9,841,378 rows and the dates span between 2026-03-01 and 2026-03-31.

Missing values: filtering gsc_data_avilable to true only gives out 36.7% of the total rows, so almost two thirds of the data does not have gsc data that I can use.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

dupes = (
    df.groupby(["client_hash_id", "content_hash_id", "report_date"])
      .size()
      .reset_index(name="c")
      .query("c > 1")
)
print("duplicate grain keys:", len(dupes))

print("rows:", len(df))
print("date span:", df["report_date"].min(), "-", df["report_date"].max())

before = len(df)
avail = df[df["gsc_data_available"] == True]
after = len(avail)
print(f"before: {before}, after gsc_data_available IS TRUE: {after} ({after/before:.1%})")


duplicate grain keys: 0
rows: 9841378
date span: 2026-03-01 - 2026-03-31
before: 9841378, after gsc_data_available IS TRUE: 3611061 (36.7%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

A significant portion of the data does not have gsc_data available so any gsc features would talk about a smaller portion of the data.
The history is also not balanced as roughly 8% of the content items were created in mid-march


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
content_dates = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]},
    columns=["client_hash_id", "content_hash_id", "content_created_date"]
)

df = df.merge(content_dates, on=["client_hash_id", "content_hash_id"], how="left")
del content_dates
import gc; gc.collect()

d_created = pd.to_datetime(df["content_created_date"], errors="coerce")
mid_march = d_created.between("2026-03-01", "2026-03-31")
print("content items created mid-window (March):", df.loc[mid_march, "content_hash_id"].nunique())
print("total unique content items in slice:", df["content_hash_id"].nunique())

content items created mid-window (March): 26536
total unique content items in slice: 331437


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.